# v13_2 Benchmark Runner — vast.ai RTX Pro 6000

Runs **v13_2 vs v13_1** head-to-head on ls20 (1200s/level) and ar25 (600s/level).
All BFS, pure CPU workers, fork context. Run cells top to bottom once.


In [1]:
# 1. Clone the repo (skip if already cloned)
import os
REPO = "/root/arc3"
if not os.path.exists(REPO):
    !git clone https://github.com/shreyasmahimkar/arc-agi-3 {REPO}
else:
    !git -C {REPO} pull --ff-only
print("Repo ready at", REPO)


Already up to date.
Repo ready at /root/arc3


In [2]:
# 2. Install arcengine + dependencies from bundled wheels
import os, sys

WHEELS = "/root/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
os.system(f"{sys.executable} -m pip install -q --no-index --find-links {WHEELS} arc-agi pydantic python-dotenv")

# Verify it actually landed in the right interpreter
os.system(f"{sys.executable} -c \"import arcengine; print('arcengine OK')\"")


arcengine OK


0

In [3]:
# 3. Verify hardware — expect many cores, fork context
import multiprocessing, platform, subprocess
ncpu = multiprocessing.cpu_count()
print(f"CPUs: {ncpu}  |  OS: {platform.system()}  |  Workers we will use: {ncpu - 1}")
try:
    r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                       capture_output=True, text=True, timeout=5)
    print("GPU:", r.stdout.strip())
except Exception:
    print("No nvidia-smi (CPU-only run is fine)")


CPUs: 256  |  OS: Linux  |  Workers we will use: 255
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition, 97887 MiB


## ls20 — 1200s/level
L5 needs this headroom. Expect ~2-3 hrs total for 7 levels × 2 versions.

In [ ]:
import os, multiprocessing, subprocess, sys

REPO = "/root/arc3"
SOLVER = os.path.join(REPO, "CommunitySolutions/chronos_solver/v13_2")
WORKERS = multiprocessing.cpu_count() - 1
print(f"Running with {WORKERS} workers")

cmd = [
    sys.executable, "benchmark.py",
    "--games", "ls20:7",
    "--versions", ".,../v13_1",
    "--budget", "1200",
    "--workers", str(WORKERS),
    "--max-states", "10000000",
    "--out", "ls20_benchmark_1200s.json",
]
subprocess.run(cmd, cwd=SOLVER, env={**os.environ, "PYTHONUNBUFFERED": "1"})

Running with 255 workers
=== v13_2 / ls20 (7 levels, 1200.0s/level) ===
[v13_2/ls20] generated new fontManager
[v13_2/ls20] [v13] hardware profile: CPU device=cpu vram=n/aGB workers=255 bsz=64 mp_ctx=fork
[v13_2/ls20] BFS L0: 4 effective actions
[v13_2/ls20] BFS: transient mask covers 128 px / rows [61, 62]
[v13_2/ls20] BFS L0: SOLVED in 13 actions (332 explored, 2.0s)
[v13_2/ls20] L0: SOLVED 13 actions in 2.8s
[v13_2/ls20] BFS L1: using TRUE chained baseline (replayed L0..L0)
[v13_2/ls20] BFS L1: 4 effective actions
[v13_2/ls20] BFS: transient mask covers 128 px / rows [61, 62]
[v13_2/ls20] BFS L1: SOLVED in 45 actions (2772 explored, 5.5s)
[v13_2/ls20] L1: SOLVED 45 actions in 7.8s
[v13_2/ls20] BFS L2: using TRUE chained baseline (replayed L0..L1)
[v13_2/ls20] BFS L2: 4 effective actions
[v13_2/ls20] BFS: transient mask covers 128 px / rows [61, 62]
[v13_2/ls20] BFS L2: SOLVED in 39 actions (3816 explored, 5.4s)
[v13_2/ls20] L2: SOLVED 39 actions in 7.9s
[v13_2/ls20] BFS L3: using TR

## ar25 — 600s/level
Space-limited, not time-limited — 600s is enough.

In [ ]:
import os, multiprocessing, subprocess, sys

REPO = "/root/arc3"
SOLVER = os.path.join(REPO, "CommunitySolutions/chronos_solver/v13_2")
WORKERS = multiprocessing.cpu_count() - 1
print(f"Running with {WORKERS} workers")

cmd = [
    sys.executable, "benchmark.py",
    "--games", "ar25:3",
    "--versions", ".,../v13_1",
    "--budget", "600",
    "--workers", str(WORKERS),
    "--max-states", "10000000",
    "--out", "ar25_benchmark_600s.json",
]
subprocess.run(cmd, cwd=SOLVER, env={**os.environ, "PYTHONUNBUFFERED": "1"})

## Results

In [ ]:
import json, os
SOLVER = "/root/arc3/CommunitySolutions/chronos_solver/v13_2"

for fname in ["ls20_benchmark_1200s.json", "ar25_benchmark_600s.json"]:
    path = os.path.join(SOLVER, fname)
    if not os.path.exists(path):
        print(f"{fname}: not found yet")
        continue
    rows = json.load(open(path))
    print(f"
=== {fname} ===")
    print(f"  {'Game':<8} {'Level':<6} {'Version':<10} {'Solved':<8} {'Actions':<10} {'Time(s)':<10}")
    for r in rows:
        solved = "YES" if r.get("solved") else "no"
        acts = r.get("actions", "-")
        t = round(r.get("elapsed", 0), 1)
        print(f"  {r['game']:<8} L{r['level']:<5} {r['version']:<10} {solved:<8} {str(acts):<10} {t}")


In [ ]:
# Print the markdown summary table written by benchmark.py
import os
SOLVER = "/root/arc3/CommunitySolutions/chronos_solver/v13_2"
for fname in ["BENCHMARK.md"]:
    p = os.path.join(SOLVER, fname)
    if os.path.exists(p):
        print(open(p).read())
